In [ ]:
!pip install -q faster-whisper edge-tts transformers accelerate bitsandbytes gradio
!pip install -q nest-asyncio # Needed for async TTS in Jupyter/Kaggle

In [ ]:
import gradio as gr
import torch
import edge_tts
import asyncio
import nest_asyncio
import re
from faster_whisper import WhisperModel
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# Allow async loops in Kaggle notebooks
nest_asyncio.apply()

# ==========================================
# 1. MOCK DATABASE & LOGIC
# ==========================================
# In a real app, this connects to Google Calendar or a SQL database
calendar_db = {
    "2026-05-20 10:00": "available",
    "2026-05-20 11:00": "booked",
}
waitlist_db = []

def manage_booking(intent, date_time, user_id="guest"):
    if intent == "book":
        if calendar_db.get(date_time) == "available":
            calendar_db[date_time] = "booked"
            return f"Success! I have booked your appointment for {date_time}."
        elif calendar_db.get(date_time) == "booked":
            waitlist_db.append((user_id, date_time))
            return f"That slot is taken. I have added you to the waitlist for {date_time}."
        else:
            return "That time slot does not exist. Please choose 10:00 or 11:00."
            
    elif intent == "cancel":
        if calendar_db.get(date_time) == "booked":
            # Check if anyone is waiting for this specific time slot
            waiting_users = [u for u, t in waitlist_db if t == date_time]
            
            if waiting_users:
                # Get the first person in line
                next_user = waiting_users[0]
                waitlist_db.remove((next_user, date_time))
                # The slot remains "booked", but conceptually it belongs to the waitlist user now
                return f"Your appointment for {date_time} is canceled. The slot has been automatically given to the next patient on the waitlist."
            else:
                # No one on waitlist, open the slot
                calendar_db[date_time] = "available"
                return f"Your appointment for {date_time} has been canceled."
                
    return "I couldn't process that calendar request."

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ==========================================
# 2. MODEL INITIALIZATION (Kaggle T4 Optimized)
# ==========================================
print("Loading Whisper STT...")
# Using 'small' model for fast transcription on T4
stt_model = WhisperModel("small", device="cuda", compute_type="float16")

print("Loading LLM (Quantized)...")
model_id = "HuggingFaceH4/zephyr-7b-beta" 

# Define the 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", # Recommended for better accuracy
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    quantization_config=bnb_config # Pass the config object here instead
)

llm_pipeline = pipeline("text-generation", model=llm_model, tokenizer=tokenizer)

# ==========================================
# 3. PIPELINE FUNCTIONS
# ==========================================
def transcribe_audio(audio_path):
    # The task="translate" parameter forces Whisper to convert the speech 
    # to English text, no matter what language is spoken.
    segments, info = stt_model.transcribe(audio_path, beam_size=5, task="translate")
    
    return " ".join([segment.text for segment in segments]), info.language

def generate_response(user_text, language):
    # System prompt forces the LLM to act as the agent and extract actionable data
    system_prompt = """You are a helpful voice assistant for a clinic. 
    If the user wants to book, cancel, or reschedule, extract the intent and the time. 
    Format your response exactly like this if an action is needed: [ACTION: book|cancel, TIME: YYYY-MM-DD HH:MM] 
    Otherwise, just respond naturally."""
    
    prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_text}\n<|assistant|>\n"
    
    outputs = llm_pipeline(prompt, max_new_tokens=100, temperature=0.3)
    response = outputs[0]["generated_text"].split("<|assistant|>\n")[-1].strip()
    return response

async def text_to_speech(text, output_filename="output.mp3"):
    # Edge-TTS handles multiple languages natively based on the voice chosen
    # Defaulting to a clear English voice, but this can be dynamic based on the STT language detected
    communicate = edge_tts.Communicate(text, "en-US-AriaNeural")
    await communicate.save(output_filename)
    return output_filename

# ==========================================
# 4. MAIN AGENT LOOP
# ==========================================
def process_voice_interaction(audio_filepath):
    if not audio_filepath:
        return None, "No audio detected."

    # 1. Speech to Text
    user_text, detected_lang = transcribe_audio(audio_filepath)
    
    # 2. LLM Processing
    llm_output = generate_response(user_text, detected_lang)
    
    # 3. Action Extraction & Booking Logic
    # Look for the structured action tag the LLM was prompted to create
    action_match = re.search(r"\[ACTION: (.*?), TIME: (.*?)\]", llm_output)
    
    if action_match:
        intent = action_match.group(1).strip()
        time_slot = action_match.group(2).strip()
        final_response_text = manage_booking(intent, time_slot)
    else:
        final_response_text = llm_output

    # 4. Text to Speech
    loop = asyncio.get_event_loop()
    output_audio = loop.run_until_complete(text_to_speech(final_response_text))

    
    # Create a clean string showing the current databases
    db_state = f"📅 Calendar State:\n{calendar_db}\n\n⏳ Waitlist:\n{waitlist_db}"
    
    # Return 3 things now: audio, transcript, and the database state
    return output_audio, f"**User:** {user_text}\n\n**Agent:** {final_response_text}", db_state


# ==========================================
# 5. GRADIO INTERFACE
# ==========================================
demo = gr.Interface(
    fn=process_voice_interaction,
    inputs=gr.Audio(sources=["microphone"], type="filepath", label="Speak to the Agent"),
    outputs=[
        gr.Audio(label="Agent Voice Response", autoplay=True),
        gr.Markdown(label="Conversation Transcript"),
        gr.Textbox(label="Real-time Database State", lines=4)
    ],
    title="🎙️ Multi-lingual AI Booking Agent",
    description="Ask to book or cancel an appointment for '2026-05-20 10:00' or '11:00'."
)

if __name__ == "__main__":
    demo.launch(share=True) # share=True generates a public link to use your microphone from your local browser